In [2]:
#| default_exp pca_module

# PCA on data divided by stature

## Import libraries

In [3]:
#| export 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA

## Plots

### Scree-plot:

Shows the procentages of variance explained by the different PCs @(PCA_review_reset)

In [4]:
#| export 
def scree_plot(data, subset: list ,subset_name: str):
    ansur = data
    #df_subset=create_subset(subset,group)
    df_subset= ansur[subset]
    pca = PCA(n_components=len(df_subset.keys()))
    principal_components = pca.fit_transform(df_subset)

    per_var = np.round(pca.explained_variance_ratio_* 100, decimals=1)
    labels = ['PC' + str(x) for x in range(1, len(per_var)+1)]
    #df = pd.DataFrame(data=d)
    df_per_var = pd.DataFrame(data=per_var).T
    df_per_var.columns=labels
    print(df_per_var)
    
    per = 0
    for i in range(len(per_var)):
        per += per_var[i]
        if per >= 95:
            print('Interesting PCs to get 95%: '+ str(labels[:i+1]))
            break
    
    plt.bar(x=range(1,len(per_var[:7])+1), height=per_var, tick_label=labels)
    plt.ylabel('Percentage of Explained Variance')
    plt.xlabel('Principal Component')
    plt.title('Scree Plot: '+subset_name)
    plt.xticks(rotation=90)
    plt.show()


### Pairplot:

Shows the correlation between all measurements. The genders are represented by different colors

In [5]:
#| export 
def Pairplot(data,data_scaled,compare:str):
    ansur = data
    ansur_scaled = data_scaled

    ansur_scaled[compare]=ansur[compare]

    sns.pairplot(data_scaled, hue=compare)

### 2D-plot:

Shows the correlation between the two first PCs

In [6]:
#| export 
def PC2_plot(data,data_scaled,subset: list,subsetname: str, compare: str,  ax =None ):
    #df_subset =male_and_female_no_mean(subset)
    df_subset = data_scaled[subset]
    ansur_c = data.copy()
    #data_name = get_var_name(data)[0]

    # PCA med 2 komponenter
    pca = PCA(n_components=len(df_subset.keys()))# =2,whiten=True)
    X_pca = pca.fit_transform(df_subset) # fit--> Beräknar PCA baserat på df_subset, transform --> Transformerar datan till de nya PC-axlarna
    

    # Skapa en DataFrame för plottning
    #df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
    df_pca = pd.DataFrame(X_pca, columns = [f'PC{i+1}' for i in range(len(df_subset.keys()))])
    #df_pca_subset.columns = [f'PC{i+1}' for i in range(len(df_subset.keys()))]
    ansur_c = ansur_c.reset_index(drop=True) # Only needed when subset?
    

    #df_pca['Gender']=ansur_c['Gender'] #SHOULD I DO LIKE THIS??????????
    df_pca[compare]=ansur_c[compare]
   

    per_var = np.round(pca.explained_variance_ratio_* 100, decimals=1)

    if ax == None:
    # Plotta PC1 vs PC2 färglagd efter kön
        plt.figure(figsize=(8,6))
        sns.scatterplot(x=df_pca["PC1"], y=df_pca["PC2"], hue=df_pca[compare], alpha=0.6, palette="coolwarm", ax=ax)
        plt.xlabel('PC1 - {0}%'.format(per_var[0]))
        plt.ylabel('PC2 - {0}%'.format(per_var[1]))
        plt.title("PCA: Difference between "+ compare +' - ' +subsetname)#+' - ' +data_name)
        plt.axhline(0, color='gray', linestyle='--')
        plt.axvline(0, color='gray', linestyle='--')
        plt.legend(title= compare , bbox_to_anchor=(1, 1))

        plt.show()

    else: 
        sns.scatterplot(x=df_pca["PC1"], y=df_pca["PC2"], hue=df_pca[compare], alpha=0.6, palette="coolwarm", ax=ax)
        ax.set_xlabel('PC1 - {0}%'.format(per_var[0]))
        ax.set_ylabel('PC2 - {0}%'.format(per_var[1]))
        ax.set_title("PCA: Difference between " + compare + ' - ' + subsetname)#+' - ' +data_name)
        ax.axhline(0, color='gray', linestyle='--')
        ax.axvline(0, color='gray', linestyle='--')
        ax.legend(title=compare, loc="upper left", bbox_to_anchor=(1, 1))

### Biplot:

Shows the 2D-plot with arrows representing the variables

In [7]:
#| export 
def PC2_plot_biplot(data,data_scaled,subset: list,subsetname: str,compare: str, ax = None ): #
    #if ax is None:
     #   ax = plt.gca()  # Använd aktuell axel om ingen ges
    ansur = data
    ansur_c=ansur.copy()
    df_subset = data_scaled[subset]
    #loadings = loadings_function(data,subset)

    # PCA med 2 komponenter
    pca = PCA(n_components=len(df_subset.keys()))#2)#,whiten=True)
    X_pca = pca.fit_transform(df_subset) # fit--> Beräknar PCA baserat på df_subset, transform --> Transformerar datan till de nya PC-axlarna
    
    # Skapa en DataFrame för plottning
    df_pca = pd.DataFrame(X_pca, columns= [f'PC{i+1}' for i in range(len(df_subset.keys()))])
    ansur_c = ansur_c.reset_index(drop=True) # Only needed when subset?
    df_pca[compare]=ansur_c[compare]

    
    loadings = pca.components_.T
    per_var = np.round(pca.explained_variance_ratio_* 100, decimals=1)

    loading_lengths = np.linalg.norm(loadings[:, :2], axis=1)  # L2-norm (dvs. längden på pilen)
    # Indices till de 10 längsta pilarna
    top_10_indices = np.argsort(loading_lengths)[-10:]  

    # Plotta PC1 vs PC2 färglagd efter kön
    #plt.figure(figsize=(8,6))

    if ax == None:
        plt.figure(figsize=(12,10))
        sns.scatterplot(x=df_pca["PC1"], y=df_pca["PC2"], hue=df_pca[compare], alpha=0.6, palette="coolwarm")
        plt.xlabel('PC1 - {0}%'.format(per_var[0]))
        plt.ylabel('PC2 - {0}%'.format(per_var[1]))
        plt.title("PCA: Difference between "+ compare +' - ' +subsetname)
        plt.axhline(0, color='gray', linestyle='--')
        plt.axvline(0, color='gray', linestyle='--')
        #loadings = df_pca.components_.T  # PCA-laddningar
        #loadings = pca.components_

        if subsetname == 'All':
            #for i, var in enumerate(df_subset.columns):
             #   plt.arrow(0, 0, loadings[i, 0]*500, loadings[i, 1]*500, color='red', alpha=0.5)
              #  plt.text(loadings[i, 0]*500, loadings[i, 1]*500, var, color='red')
            for i in top_10_indices:
                plt.arrow(0, 0, loadings[i, 0]*0.006, loadings[i, 1]*0.006, color='red', alpha=0.5, width=0.00003)
                plt.text(loadings[i, 0]*0.006+0.0001, loadings[i, 1]*0.006, df_subset.columns[i], color='red') 
        elif subsetname == 'Torso':
            for i in top_10_indices:
                plt.arrow(0, 0, loadings[i, 0]*0.005, loadings[i, 1]*0.005, color='red', alpha=0.5, width=0.00002)
                plt.text(loadings[i, 0]*0.005+0.0001, loadings[i, 1]*0.005, df_subset.columns[i], color='red') 
        else:
            for i, var in enumerate(df_subset.columns):
                plt.arrow(0, 0, loadings[i, 0]*0.0003, loadings[i, 1]*0.0003, color='red', alpha=0.5, width=0.000002)
                plt.text(loadings[i, 0]*0.0003, loadings[i, 1]*0.0003, var, color='red')
        
        plt.legend(title= compare,loc="upper left", bbox_to_anchor=(1, 1))
        plt.grid()
        plt.show()
    
    else: 
        sns.scatterplot(x=df_pca["PC1"], y=df_pca["PC2"], hue=df_pca[compare], alpha=0.6, palette="coolwarm",ax=ax)
        ax.set_xlabel('PC1 - {0}%'.format(per_var[0]))
        ax.set_ylabel('PC2 - {0}%'.format(per_var[1]))
        ax.set_title("PCA: Difference between "+ compare +' - ' +subsetname)
        ax.axhline(0, color='gray', linestyle='--')
        ax.axvline(0, color='gray', linestyle='--')
        #loadings = df_pca.components_.T  # PCA-laddningar
        loadings = pca.components_
        if subsetname == 'All':
            for i, var in enumerate(df_subset.columns):
                ax.arrow(0, 0, loadings[i, 0]*0.003, loadings[i, 1]*0.003, color='red', alpha=0.5,width=0.00005)
                ax.text(loadings[i, 0]*0.003, loadings[i, 1]*0.003, var, color='red')
        else:
            for i, var in enumerate(df_subset.columns):
                #ax.arrow(0, 0, loadings[i, 0]*df_subset[var].mean(), loadings[i, 1]*df_subset[var].mean(), color='red', alpha=0.5)
                #ax.text(loadings[i, 0]*df_subset[var].mean(), loadings[i, 1]*df_subset[var].mean(), var, color='red')
                ax.arrow(0, 0, loadings[i, 0]*0.0003, loadings[i, 1]*0.0003, color='red', alpha=0.5,width=0.00001)
                ax.text(loadings[i, 0]*0.0003, loadings[i, 1]*0.0003, var, color='red')
                    
            ax.legend(title= compare,loc="upper left", bbox_to_anchor=(1, 1))


### Loadings:

Function thet gives the loading for the PCs from the PCA

In [8]:
#| export 
def loadings_function(data, subset: list):
    ansur = data.copy()
    df_subset = ansur[subset]
    
    #Gives the PCA for all the measurments in the subset for the data
    pca_subset = PCA(n_components=len(df_subset.keys()))
    principal_components = pca_subset.fit_transform(df_subset)

    # Create a DataFrame with the information of the loadings
    df_loadings_subset = pd.DataFrame(pca_subset.components_, columns=df_subset.keys(), index=[f'PC{i+1}' for i in range(len(df_subset.keys()))])
    return df_loadings_subset

Shows a barplot of the top 10 most influencing loadings and their values

Here "number" is number of the PC that will be plotted

In [9]:
#| export 
def plot_loadings(data,subset: list,subsetname:str, number:int,ax=None):
    df_loadings = loadings_function(data,subset)
    df_loadings_val=df_loadings.T

    #top_10_values = df_loadings_val.nlargest(10, f'PC{number}')
    top_10_values = df_loadings_val.loc[df_loadings_val[f'PC{number}'].abs().nlargest(10).index]


    #if ax == None: 
    if ax is None:
        #sns.barplot(x=df_loadings.keys(), y=df_loadings_val[f'PC{number}'])
        sns.barplot(x=top_10_values.T.keys(), y=top_10_values[f'PC{number}'])
        plt.title(f'Value of top {len(top_10_values.T.keys())} loadings with biggest influence - {subsetname}')
        plt.ylabel(f'PC{number}')
        plt.xlabel('Measurement')
        plt.axhline(0, color='black', linewidth=0.8)  # Lägg till en linje vid y=0
        plt.xticks(rotation=90)
        plt.show()
    else:
        sns.barplot(x=top_10_values.T.keys(), y=top_10_values[f'PC{number}'],ax=ax)
        # Anpassa plotten
        ax.set_title(f'Value of top {len(top_10_values.T.keys())} loadings with biggest influence - {subsetname}')
        ax.set_ylabel(f'PC{number}')
        ax.set_xlabel('Measurement')
        ax.axhline(0, color='black', linewidth=0.8)  # Lägg till en linje vid y=0
        #ax.set_xticks(range(len(df_loadings.keys())))
        #ax.set_xticklabels(ax,rotation=90)


In [10]:
import nbdev; nbdev.nbdev_export()